# SmolVLA × LIBERO-Goal — Baseline Eval (Colab, free T4)

Zero-shot baseline of the **official LIBERO-finetuned** checkpoint
`HuggingFaceVLA/smolvla_libero` on **LIBERO-Goal**, run top-to-bottom on a free T4.

**Read before running:**
- ⏱️ **90-min idle timeout** and **~12-hr session cap**. Don't walk away mid-eval.
- 🎲 Colab does **not guarantee** a GPU model; free tier is usually a T4 (16GB) — plenty for inference + sim.
- 🎯 **Expected LIBERO-Goal range: ~78–84%** (reproduced), not the paper's 92%. Report what you get.
- 🛑 **0% success** almost always means one of two things — `--env.control_mode` (relative vs absolute) or an image-key mismatch (`--rename_map`). Cells below check both.
- 💾 Outputs go to **Google Drive** so a disconnect doesn't lose them.
- Every cell is **re-runnable**; re-run from the top after a reconnect.

## 1. GPU check

In [1]:
!nvidia-smi
import sys; print('python', sys.version)
# LeRobot v0.5 needs Python 3.12+. If Colab's python is older, switch runtime
# or use a 3.12 image; the pip install below will warn/fail otherwise.

/bin/bash: line 1: nvidia-smi: command not found
python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


## 2. Install pinned environment

Installs ffmpeg + mesa/EGL system libs, then `lerobot[smolvla,libero]`.
First run takes several minutes and burns some compute units (expected).

In [2]:
# System libs for headless MuJoCo rendering (EGL + OSMesa fallback) and ffmpeg.
!sudo apt-get -qq update
!sudo apt-get -qq install -y ffmpeg libegl1-mesa-dev libgl1-mesa-glx libosmesa6-dev libglfw3 > /dev/null
print('apt deps installed')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 12.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
apt deps installed


In [3]:
# LeRobot with smolvla + libero extras. Pin is recorded post-install below.
%pip install -q "lerobot[smolvla,libero]" 2>&1 | tail -n 5
print('lerobot install step done')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.2/182.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.7/51.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 6.8 MB/s eta 0:00:00
lerobot install step done


In [4]:
# Record the EXACT resolved versions (reproducibility).
import importlib, platform
def v(m):
    try: return importlib.import_module(m).__version__
    except Exception as e: return f'ERR {e}'
import torch
print('python      ', platform.python_version())
print('lerobot     ', v('lerobot'))
print('torch       ', torch.__version__, '| cuda', torch.version.cuda, '| avail', torch.cuda.is_available())
print('transformers', v('transformers'))

python       3.12.13
lerobot      0.5.1
torch        2.10.0+cu128 | cuda 12.8 | avail False
transformers 5.3.0


## 3. Headless rendering backend
Set **before** importing mujoco/libero. EGL = GPU headless (preferred on Colab).

In [5]:
import os
os.environ['MUJOCO_GL'] = 'egl'   # fallback: 'osmesa' (CPU) if egl errors
os.environ['HF_HOME'] = '/content/hf_cache'
print('MUJOCO_GL =', os.environ['MUJOCO_GL'])

MUJOCO_GL = egl


## 4. Mount Google Drive (persist outputs)

In [6]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
OUT_DIR = '/content/drive/MyDrive/VLATune/baseline_eval'
pathlib.Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print('outputs ->', OUT_DIR)

Mounted at /content/drive
outputs -> /content/drive/MyDrive/VLATune/baseline_eval


## 5. Hugging Face login + W&B note

`HuggingFaceVLA/smolvla_libero` is public, so login is usually optional — only
needed if you hit rate limits. **Do not paste your token into a saved cell.**
W&B is not required for eval; if you use it later, run `wandb login` in a
throwaway cell and **never commit the key**.

In [7]:
# Optional, non-persisted. Skip if the model downloads fine without it.
# from huggingface_hub import login; from getpass import getpass
# login(getpass('HF token (leave blank to skip): ') or None)
print('skip unless you hit download rate limits')

skip unless you hit download rate limits


## 6. Reconcile CLI flags against the INSTALLED version
The installed `--help` is the source of truth. Confirm the flags we use exist.

In [8]:
help_txt = get_ipython().getoutput('lerobot-eval --help')
help_txt = '\n'.join(help_txt)
for f in ['policy.path','env.type','env.task','control_mode','max_parallel_tasks',
          'n_episodes','batch_size','rename_map','output_dir','n_action_steps','task_ids']:
    print(('FOUND   ' if f in help_txt else 'MISSING '), '--'+f)
print('\n--- if MISSING, the installed name wins; adjust the eval cell below ---')
print('--- (task_ids may be absent — if so, leave TASK_IDS empty and run all tasks) ---')

MISSING  --policy.path
FOUND    --env.type
FOUND    --env.task
FOUND    --control_mode
FOUND    --max_parallel_tasks
FOUND    --n_episodes
FOUND    --batch_size
FOUND    --rename_map
FOUND    --output_dir
FOUND    --n_action_steps
FOUND    --task_ids

--- if MISSING, the installed name wins; adjust the eval cell below ---
--- (task_ids may be absent — if so, leave TASK_IDS empty and run all tasks) ---


## 7. Render smoke test (one frame)
Proves MuJoCo/EGL works here before spending time on a 50-episode eval.

In [9]:
import os, numpy as np
from PIL import Image
ok = False
for backend in ['egl', 'osmesa']:
    try:
        os.environ['MUJOCO_GL'] = backend
        from libero.libero import benchmark, get_libero_path
        from libero.libero.envs import OffScreenRenderEnv
        suite = benchmark.get_benchmark_dict()['libero_goal']()
        task = suite.get_task(0)
        bddl = os.path.join(get_libero_path('bddl_files'), task.problem_folder, task.bddl_file)
        env = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=256, camera_widths=256)
        env.seed(0); obs = env.reset()
        frame = np.asarray(obs['agentview_image'])[::-1]
        p = os.path.join(OUT_DIR, f'render_smoke_{backend}.png')
        Image.fromarray(frame).save(p); env.close()
        print(f'RENDER OK via {backend}: {frame.shape}, non-black={frame.any()}, saved {p}')
        ok = True; break
    except Exception as e:
        print(f'render via {backend} failed: {type(e).__name__}: {e}')
if not ok:
    raise SystemExit('Render failed on both backends — fix before eval (check GPU/EGL).')

The full path of the custom storage path you entered is:
/content/datasets
Do you want to continue? (Y/N)
Initializing the default config file...
The following information is stored in the config file: /root/.libero/config.yaml
benchmark_root: /usr/local/lib/python3.12/dist-packages/libero/libero
bddl_files: /usr/local/lib/python3.12/dist-packages/libero/libero/./bddl_files
init_states: /usr/local/lib/python3.12/dist-packages/libero/libero/./init_files
datasets: /usr/local/lib/python3.12/dist-packages/libero/libero/../datasets
assets: /usr/local/lib/python3.12/dist-packages/libero/libero/./assets


[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.12/dist-packages/robosuite/scripts/setup_macros.py (__init__.py:9)


[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Local assets not found. Downloading from HuggingFace Hub...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/

Fetching 586 files:   0%|          | 0/586 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1856: DeprecationWarning: hf_xet.download_files() is deprecated. Use XetSession().new_file_download_group().start_download_file() instead.
  xet_get(


Assets downloaded successfully to /root/.cache/libero/assets


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

RENDER OK via egl: (256, 256, 3), non-black=True, saved /content/drive/MyDrive/VLATune/baseline_eval/render_smoke_egl.png


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 8. Baseline eval — `HuggingFaceVLA/smolvla_libero` on LIBERO-Goal

**Gate:** cell 6 must print `avail True` first. On a CPU runtime the policy
(`device='cuda'`) crashes or crawls — pick a **GPU + High-RAM** runtime.

**Pro+ path (current):** L4 or A100, High-RAM, **background execution ON** so the
run survives a disconnect. Run all 10 tasks at once (`TASK_IDS=''`, `BATCH_SIZE=4`).

**Official protocol:** `N_EPISODES=10` × 10 tasks = 100 episodes. Validated knobs:
`control_mode=relative`, **`n_action_steps=10` (REQUIRED — ~100s/ep vs ~400s/ep)**,
no `rename_map`. Expect a result in **~78–84%**. Wall-time: L4 ~1.5–2.8 h,
A100@batch8 ~0.7–1.4 h.

If this returns **~0% success**, set `CONTROL_MODE='absolute'` and re-run.
`RENAME_MAP` stays empty unless a key mismatch crashes the load (the
`HuggingFaceVLA` data + libero env already use `observation.images.image`/`image2`).

In [ ]:
POLICY        = 'HuggingFaceVLA/smolvla_libero'
SUITE         = 'libero_goal'
N_EPISODES    = 10          # 10 episodes/task x 10 tasks = 100 episodes (official protocol)
# Colab Pro+ with a HIGH-RAM L4/A100: run all 10 tasks at once (free-T4 chunking
# no longer needed). If you somehow still hit -9, chunk: '0,1,2' '3,4,5' '6,7,8' '9'.
TASK_IDS      = ''          # '' = all 10 Goal tasks in one run
CONTROL_MODE  = 'relative'  # flip to 'absolute' ONLY if success ~0%
N_ACTION_STEPS= 10          # REQUIRED for speed (~100s/ep vs ~400s/ep without). Keep this.
BATCH_SIZE    = 4           # L4/A100 high-RAM safe. Try 8 on A100 to ~halve wall-time.
RENAME_MAP    = ''          # e.g. '{"observation.images.OLD":"observation.images.NEW"}'
HEARTBEAT_S   = 120         # print one "still alive" line at most every N seconds

# Per-chunk output dir so chunks don't overwrite each other; '' (all) -> 'all'.
_tag     = (TASK_IDS.replace(',', '-') if TASK_IDS.strip() else 'all')
EVAL_OUT = os.path.join(OUT_DIR, f'smolvla_libero_goal_tasks{_tag}')

cmd = [
  'lerobot-eval',
  f'--policy.path={POLICY}',
  '--env.type=libero',
  f'--env.task={SUITE}',
  f'--env.control_mode={CONTROL_MODE}',
  '--env.max_parallel_tasks=1',
  f'--policy.n_action_steps={N_ACTION_STEPS}',
  f'--eval.batch_size={BATCH_SIZE}',
  f'--eval.n_episodes={N_EPISODES}',
  f'--output_dir={EVAL_OUT}',
]
if TASK_IDS.strip():
    cmd.append(f'--env.task_ids=[{TASK_IDS}]')   # draccus list syntax: [0,1,2] NOT 0,1,2
if RENAME_MAP.strip():
    cmd.append(f"--rename_map={RENAME_MAP}")
print('RUNNING:\n  ' + ' '.join(cmd) + '\n')

# Quiet stream: only echo meaningful lines; drop per-step tqdm spam; heartbeat so
# you know it's alive. FULL raw log is tee'd to Drive (eval_console.log) regardless.
import subprocess, sys, os, time
os.makedirs(EVAL_OUT, exist_ok=True)
LOG  = os.path.join(EVAL_OUT, 'eval_console.log')
KEEP = ('episode', 'success', 'making environment', 'output dir', 'aggregat',
        'average', 'avg', 'eval_s', 'error', 'traceback', 'exception', 'killed')
DROP = ('it/s', '%|', 'deprecat', 'warn')   # tqdm bars + noise
start = time.time(); last_beat = start; quiet = 0
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
with open(LOG, 'w') as logf:
    for line in proc.stdout:
        logf.write(line); logf.flush()                 # full log, always
        low = line.lower()
        if any(k in low for k in KEEP) and not any(d in low for d in DROP):
            sys.stdout.write(line); sys.stdout.flush(); quiet = 0
        else:
            quiet += 1
        now = time.time()
        if now - last_beat >= HEARTBEAT_S:
            mins = int((now - start) / 60)
            sys.stdout.write(f'... still running — {mins} min elapsed, {quiet} quiet lines since last shown\n')
            sys.stdout.flush(); last_beat = now
proc.wait()
print('\nexit code:', proc.returncode, f'| full log: {LOG}')
if proc.returncode != 0:
    print('Non-zero exit. If -9: host-RAM OOM -> fewer tasks per chunk. Else check flags vs cell 6 / the full log.')

## 9. Parse results → per-task + aggregate success, write JSON to Drive

In [11]:
import glob, json, os, datetime
# lerobot writes an eval results JSON under output_dir; find it defensively.
cands = sorted(glob.glob(os.path.join(EVAL_OUT, '**', '*.json'), recursive=True))
print('result json candidates:', cands)
data = None
for c in cands:
    try:
        d = json.load(open(c))
        if any('success' in str(k).lower() for k in (d.keys() if isinstance(d, dict) else [])):
            data = d; src = c; break
    except Exception: pass
    data = data or (d if 'd' in dir() else None)
summary = {
    'policy': POLICY, 'suite': SUITE, 'n_episodes': N_EPISODES,
    'task_ids': TASK_IDS or 'all',
    'control_mode': CONTROL_MODE, 'n_action_steps': N_ACTION_STEPS,
    'batch_size': BATCH_SIZE, 'rename_map': RENAME_MAP,
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'raw_source': (src if 'src' in dir() else None),
    'raw': data,
    'expected_range': '78-84% (reproduced); paper=92%',
}
out = os.path.join(OUT_DIR, 'baseline_libero_goal_summary.json')
json.dump(summary, open(out, 'w'), indent=2, default=str)
print('wrote', out)
print(json.dumps(data, indent=2, default=str)[:2000] if data else 'No JSON parsed — inspect EVAL_OUT manually.')

result json candidates: []
wrote /content/drive/MyDrive/VLATune/baseline_eval/baseline_libero_goal_summary.json
No JSON parsed — inspect EVAL_OUT manually.


## 10. Troubleshooting

| Symptom | Fix |
|---|---|
| 0% success, arm drifts/overshoots | set `CONTROL_MODE='absolute'`, re-run cell 8 |
| crash: key/shape mismatch on load | set `RENAME_MAP` (cell 8), see `docs/rename_map_notes.md` |
| 0% success, arm barely moves | check `observation.state` is 8-dim / wrong suite |
| `lerobot-eval: flag not recognized` | reconcile with cell 6, the installed name wins |
| render all-black | EGL can't see GPU → cell 7 already falls back to osmesa |
| ~78–84% Goal | ✅ working as intended — this is your baseline |

Re-run from the top after any reconnect. Drive keeps your outputs.